# Module 11 · Differential expression

Pseudobulk, donor-blocked, across the four quadrant contrasts.

**Donor as replicate, not cell.** Cells from one donor are not independent
observations. Aggregating to one profile per donor per population before
testing is what keeps the p-values interpretable; the alternative inflates the
effective n by the number of cells and produces significance everywhere.

**The four contrasts** derive from `AXIS`, so their names change with it:

```
<X>axis_SnCpos    axis effect within senescent cells
<X>axis_SnCneg    axis effect within non-senescent cells
SnCaxis_<X>pos    senescence effect within axis-high cells
SnCaxis_<X>neg    senescence effect within axis-low cells
```

The first two isolate activation at fixed senescence; the second two isolate
senescence at fixed activation. Comparing them is what separates the two
programmes at the gene level.

| Section | |
|---|---|
| 01-02 | config, pseudobulk build and pairing QC |
| 03 | four-contrast limma-voom loop |
| 04-06 | volcanoes, DEG counts, Venns |
| 07-08 | genome-maintenance heatmap |
| 09 | export rankings for module 12 |

**Design.** `~ 0 + pop + grp2 + Sex + Mean_Log_Library_Depth_scaled + Cohort`,
coefficient `popTEST`, minimum 10 cells per pseudobulk sample.

> **Legacy naming inside the lifted code.** Local variables and label strings
> still read `dam_z`, `SenHi_DAMlo`, `DAMaxis_*` — that is the source notebook's
> vocabulary from when the axis was fixed to DAM. The *logic* is not: those
> variables are now computed from `X_COL`, which resolves from `AXIS`. Setting
> `AXIS <- "IRM"` analyses IRM; only the variable names and some figure labels
> still say DAM. Output filenames are deliberately left alone so existing
> `DAMaxis_*` results on disk stay findable.


---
## 01 · Config

**Why.** Self-contained — the notebook carries its own config rather than
importing one, so it can be read and run without tracing an import elsewhere.
Paths come from the environment; see `.env.example`.

**`AXIS` is the one thing you change.** Column names, quadrant labels, figure
titles, contrast names and output directories all derive from it. Outputs are
namespaced by axis so two runs never overwrite each other.

```
AXIS <- "IRM"    # "IRM" | "DAM_like" | "ARM" | "Stress"
```

`STATE_ORDER` and `STATE_COLORS` list all five states regardless of `AXIS` —
those are the annotation registry, not a per-run choice.

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================
# Paths come from the environment - see .env.example. Nothing below hardcodes
# a filesystem location.
#   SENESCENCE_DATA : analysis root (module outputs written under it)
#   SENESCENCE_REF  : reference root (published panels, read-only)
# =============================================================================

SCRATCH <- Sys.getenv("SENESCENCE_DATA")
REF_DIR <- Sys.getenv("SENESCENCE_REF")
if (SCRATCH == "" || REF_DIR == "")
    stop("SENESCENCE_DATA and SENESCENCE_REF must be set. See .env.example.")


# ─────────────────────────────────────────────────────────────────────────────
# Libraries
# ─────────────────────────────────────────────────────────────────────────────
suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(readxl)
    library(ggplot2)
    library(patchwork)
    library(scales)
    library(qs)
    library(jsonlite)
    library(lme4)
    library(lmerTest)
    library(MASS)
    library(robustbase)
    library(broom)
    library(broom.mixed)
})

# Fix MASS::select masking dplyr::select
select <- dplyr::select


# ─────────────────────────────────────────────────────────────────────────────
# Inline plotting viewport (Jupyter / IRkernel)
# ─────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────
# Run parameters — edit these
# ─────────────────────────────────────────────────────────────────────────────
TISSUE     <- "brain"
STUDY_TYPE <- "disease"
DISEASE    <- "AD"
DATASET    <- "psychad_ad"

CELL_TYPE  <- "Microglia"


# ─────────────────────────────────────────────────────────────────────────────
# Stratification
# ─────────────────────────────────────────────────────────────────────────────
STRATIFY_BY_GROUP     <- TRUE
STRATIFICATION_GROUPS <- c("Old_AD", "Old_Healthy_Control")


# ─────────────────────────────────────────────────────────────────────────────
# Statistical parameters
# ─────────────────────────────────────────────────────────────────────────────
STATISTICAL_PARAMS <- list(
    min_cells_per_group  = 5L,
    fdr_threshold        = 0.05,
    fdr_method           = "BH",
    bootstrap_n_iter     = 100L,
    bootstrap_seed       = 42L,
    seed                 = 42L,
    confidence_level     = 0.95,
    rationale            = "M09-equivalent thresholds with M05 organizational rewrite"
)

set.seed(STATISTICAL_PARAMS$seed)


# ─────────────────────────────────────────────────────────────────────────────
# Derived condition tags
# ─────────────────────────────────────────────────────────────────────────────
IS_AGING          <- (STUDY_TYPE == "aging")
IS_DISEASE        <- (STUDY_TYPE == "disease")

BASE_M04   <- file.path(SCRATCH, TISSUE, "module_04T_tissue_export",
                        CONDITION_SUBPATH, DATASET)
BASE_M05   <- file.path(SCRATCH, TISSUE, "module_05_senescence_enrichment",
                        CONDITION_SUBPATH, DATASET, CELL_TYPE)

PATHS <- list(
    m04_root       = BASE_M04,
    m04_seurat     = file.path(BASE_M04, paste0(DATASET, "_tissue_seurat.qs")),
    m04_manifest   = file.path(BASE_M04, "manifest.json"),
    output_root    = BASE_M05,
    data           = file.path(BASE_M05, "data"),
    results        = file.path(BASE_M05, "results"),
    figures        = file.path(BASE_M05, "figures"),
    logs           = file.path(BASE_M05, "_logs"),
    scored_qs      = file.path(BASE_M05, "data",
                               paste0(CELL_TYPE, "_scored.qs")),
    gene_lists_rds = file.path(BASE_M05, "data", "gene_lists.rds"),
    manifest       = file.path(BASE_M05, "_logs", "m05_manifest.json"),
    markers_dir    = file.path(REF_DIR, "markers"),
    sloan_xlsx     = file.path(REF_DIR, "markers", "1-s2.0-S2666979X25003830-mmc10.xlsx"),
    senmayo_xlsx   = file.path(REF_DIR, "markers", "41467_2022_32552_MOESM4_ESM.xlsx"),
    fridman_gmt    = file.path(REF_DIR, "markers", "FRIDMAN_SENESCENCE_UP.v2026.1.Hs.gmt")
)

for (key in c("data", "results", "figures", "logs")) {
    dir.create(PATHS[[key]], recursive = TRUE, showWarnings = FALSE)
}


# ─────────────────────────────────────────────────────────────────────────────
# Color palettes
# ─────────────────────────────────────────────────────────────────────────────
LINEAGE_COLORS_BY_TISSUE <- list(
    brain = c(
        Excitatory      = "#0072B2",
        Inhibitory      = "#E69F00",
        Astrocyte       = "#009E73",
        Oligodendrocyte = "#56B4E9",
        Microglia       = "#D55E00",
        OPC             = "#CC79A7",
        Endothelial     = "#7F7F7F",
        Pericyte        = "#999999",
        VLMC            = "#A9A9A9",
        VSMC            = "#696969",
        PVM             = "#FF6347",
        Adaptive        = "#FFD700"
    ),
    pbmc = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", unconvT = "#BAB0AC",
        nkcell = "#59A14F", cd14mono = "#F28E2B", cd16mono = "#FFBE7D",
        memB = "#B07AA1", naiveB = "#76B7B2", dc = "#9C755F"
    ),
    csf = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", nkcell = "#59A14F",
        monocyte = "#F28E2B", bcell = "#B07AA1", dc = "#76B7B2"
    )
)
LINEAGE_COLORS <- LINEAGE_COLORS_BY_TISSUE[[TISSUE]]

SNC_COLORS <- c(
    Senescent       = "#C44E52",
    `Non-senescent` = "#D3D3D3",
    `TRUE`          = "#C44E52",
    `FALSE`         = "#D3D3D3",
    True            = "#C44E52",
    False           = "#D3D3D3"
)

STUDY_GROUP_COLORS <- c(
    Age_20_29 = "#2E86AB", Age_30_39 = "#4A90E2", Age_40_49 = "#50C878",
    Age_50_59 = "#FFB347", Age_60_69 = "#FF8C00", Age_70_79 = "#E24A4A",
    Age_80_100 = "#8B0000",
    Control = "#4E79A7", MCI = "#F28E2B", AD = "#E15759",
    Young_Healthy_Control = "#4A90E2",
    Old_Healthy_Control   = "#4E79A7",
    Old_AD                = "#E15759",
    All                   = "#7F7F7F"
)

PHASE_COLORS <- c(G1 = "#4E79A7", S = "#F28E2B", G2M = "#E15759")

SEX_COLORS <- c(
    Male = "#5D6D7E", Female = "#A569BD",
    M    = "#5D6D7E", F      = "#A569BD"
)

MODEL_AGREEMENT_COLORS <- c(
    `Up (sig)`     = "#C44E52",
    `Up (ns)`      = "#F4B5B5",
    `ns`           = "#D3D3D3",
    `Down (ns)`    = "#A8C5DC",
    `Down (sig)`   = "#3B6F8F"
)


# ─────────────────────────────────────────────────────────────────────────────
# Plot style
# ─────────────────────────────────────────────────────────────────────────────
PLOT_STYLE <- list(
    dpi        = 150,
    dpi_save   = 300,
    font_size  = 10,
    title_size = 11,
    formats    = c("pdf", "png", "svg"),
    pt_size    = 0.05,
    label_size = 4
)

theme_clean <- function(base_size = PLOT_STYLE$font_size) {
    theme_classic(base_size = base_size) +
    theme(
        plot.title       = element_text(size = PLOT_STYLE$title_size,
                                        face = "plain", hjust = 0),
        legend.title     = element_text(size = base_size, face = "plain"),
        panel.border     = element_rect(color = "black", fill = NA, linewidth = 0.5),
        panel.grid       = element_blank(),
        axis.line        = element_blank()
    )
}


# ─────────────────────────────────────────────────────────────────────────────
# Formatting helpers
# ─────────────────────────────────────────────────────────────────────────────
fmt_n <- function(n) format(round(n), big.mark = ",", scientific = FALSE)

fmt_size <- function(path) {
    if (!file.exists(path)) return("missing")
    sz <- file.size(path)
    if (sz > 1024^3) return(sprintf("%.2f GB", sz / 1024^3))
    if (sz > 1024^2) return(sprintf("%.1f MB", sz / 1024^2))
    sprintf("%.1f KB", sz / 1024)
}

fmt_pct <- function(num, denom) {
    if (denom == 0) return(sprintf("%s (--)", fmt_n(num)))
    sprintf("%s (%.1f%%)", fmt_n(num), num / denom * 100)
}

fmt_elapsed <- function(secs) {
    if (secs < 60)   return(sprintf("%.1f sec", secs))
    if (secs < 3600) return(sprintf("%.1f min", secs / 60))
    sprintf("%.1f hr", secs / 3600)
}

fmt_p <- function(p) {
    if (is.na(p)) return("NA")
    if (p < 0.001) return(sprintf("%.2e", p))
    sprintf("%.3f", p)
}

fmt_p_short <- function(p) {
    if (is.na(p)) return("--")
    if (p < 0.001) return(sprintf("%.1e", p))
    sprintf("%.3f", p)
}

sig_stars <- function(p) {
    ifelse(is.na(p), "",
    ifelse(p < 0.001, "***",
    ifelse(p < 0.01,  "**",
    ifelse(p < 0.05,  "*", "ns"))))
}

now_iso <- function() format(Sys.time(), "%Y-%m-%dT%H:%M:%S")

bytes_str <- function(x) format(x, scientific = FALSE, trim = TRUE)


# ─────────────────────────────────────────────────────────────────────────────
# Color helpers
# ─────────────────────────────────────────────────────────────────────────────
text_color_for_bg <- function(hex) {
    rgb_vals  <- col2rgb(hex)
    luminance <- 0.299 * rgb_vals[1, ] + 0.587 * rgb_vals[2, ] + 0.114 * rgb_vals[3, ]
    ifelse(luminance < 140, "white", "black")
}


# ─────────────────────────────────────────────────────────────────────────────
# Time / save helpers
# ─────────────────────────────────────────────────────────────────────────────
time_step <- function(label, expr) {
    cat(sprintf("\n▸ %s\n", label))
    t0  <- Sys.time()
    res <- expr
    elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
    cat(sprintf("  ✓ %s  (%s)\n", label, fmt_elapsed(elapsed)))
    res
}

save_figure <- function(fig, slug, width = 10, height = 7) {
    for (ext in PLOT_STYLE$formats) {
        path <- file.path(PATHS$figures, paste0(slug, ".", ext))
        ggsave(path, fig, width = width, height = height,
               dpi = PLOT_STYLE$dpi_save, bg = "white")
    }
    cat(sprintf("  ✓ saved → figures/%s.{%s}\n",
                slug, paste(PLOT_STYLE$formats, collapse = ",")))
}

save_table <- function(df, slug, row.names = FALSE) {
    path <- file.path(PATHS$results, paste0(slug, ".csv"))
    write.csv(df, path, row.names = row.names)
    cat(sprintf("  ✓ saved → results/%s.csv  (%d rows)\n",
                slug, nrow(df)))
}


# ─────────────────────────────────────────────────────────────────────────────
# filter_to_stratum() — slice metadata to one stratum
# ─────────────────────────────────────────────────────────────────────────────
filter_to_stratum <- function(md, stratum, study_group_col) {
    if (stratum == "All") return(md)
    md[md[[study_group_col]] == stratum, , drop = FALSE]
}


# ─────────────────────────────────────────────────────────────────────────────
# tidy_model_results() — standardize one-row result records across all models
# ─────────────────────────────────────────────────────────────────────────────
tidy_model_results <- function(stratum, outcome, model,
                               n_donors, n_cells_test, n_cells_ref,
                               estimate, se, ci_low, ci_high,
                               statistic, p_value,
                               extra = NULL) {
    out <- data.frame(
        stratum      = as.character(stratum),
        outcome      = as.character(outcome),
        model        = as.character(model),
        n_donors     = as.integer(n_donors),
        n_cells_test = as.integer(n_cells_test),
        n_cells_ref  = as.integer(n_cells_ref),
        estimate     = as.numeric(estimate),
        se           = as.numeric(se),
        ci_low       = as.numeric(ci_low),
        ci_high      = as.numeric(ci_high),
        statistic    = as.numeric(statistic),
        p_value      = as.numeric(p_value),
        stringsAsFactors = FALSE
    )
    if (!is.null(extra) && length(extra) > 0) {
        for (nm in names(extra)) {
            v <- extra[[nm]]
            if (length(v) != 1) v <- I(list(v))
            out[[nm]] <- v
        }
    }
    out
}


# ─────────────────────────────────────────────────────────────────────────────
# Library version log
# ─────────────────────────────────────────────────────────────────────────────
R_PKG_VERSIONS <- list(
    R          = R.version$version.string,
    Seurat     = as.character(packageVersion("Seurat")),
    Matrix     = as.character(packageVersion("Matrix")),
    dplyr      = as.character(packageVersion("dplyr")),
    tidyr      = as.character(packageVersion("tidyr")),
    readxl     = as.character(packageVersion("readxl")),
    ggplot2    = as.character(packageVersion("ggplot2")),
    patchwork  = as.character(packageVersion("patchwork")),
    qs         = as.character(packageVersion("qs")),
    jsonlite   = as.character(packageVersion("jsonlite")),
    lme4       = as.character(packageVersion("lme4")),
    lmerTest   = as.character(packageVersion("lmerTest")),
    MASS       = as.character(packageVersion("MASS")),
    robustbase = as.character(packageVersion("robustbase")),
    broom      = as.character(packageVersion("broom")),
    broom.mixed = as.character(packageVersion("broom.mixed"))
)



# =============================================================================
# §0.2 — AXIS SELECT
# =============================================================================
# The ONE thing you change to re-run the whole flow on a different state.
# Everything downstream — column names, quadrant labels, figure titles,
# legends, output paths — derives from this. Nothing is hardcoded per state.
# =============================================================================

AXIS <- "IRM"          # <<< "IRM" | "DAM_like" | "ARM" | "Stress"

# --- registry: score column candidates + display label + canonical hex -------
# Score columns are looked up in order; the first present on the object wins.
AXIS_REGISTRY <- list(
    # tag = short token used in CONTRAST NAMES and therefore in GSEA/DE filenames.
    # It is deliberately NOT the same as `lab` (display) or the list key: existing
    # results on disk are named DAMaxis_*, not DAM_likeaxis_*.
    IRM      = list(cols = c("Score_IRM",      "IRM1"),      lab = "IRM",      tag = "IRM",    hex = "#2980B9"),
    DAM_like = list(cols = c("Score_DAM_like", "DAM_like1"), lab = "DAM-like", tag = "DAM",    hex = "#C0392B"),
    ARM      = list(cols = c("Score_ARM",      "ARM1"),      lab = "ARM",      tag = "ARM",    hex = "#E67E22"),
    Stress   = list(cols = c("Score_Stress",   "Stress1"),   lab = "Stress",   tag = "Stress", hex = "#8E44AD")
)
stopifnot(AXIS %in% names(AXIS_REGISTRY))

AXIS_SPEC <- AXIS_REGISTRY[[AXIS]]
X_LAB     <- AXIS_SPEC$lab
X_HEX     <- AXIS_SPEC$hex
Y_LAB     <- "Senescence"
Y_TAG     <- "SnC"
X_TAG     <- AXIS_SPEC$tag
SEN_CANDIDATES <- c("senescence_score", "SenePy_score")

# --- quadrant labels derive from the axis -----------------------------------
QUAD_COL    <- paste0("quad4_", tolower(AXIS))
QUAD_LEVELS <- c(sprintf("Sen- %s-", X_LAB), sprintf("Sen+ %s-", X_LAB),
                 sprintf("Sen- %s+", X_LAB), sprintf("Sen+ %s+", X_LAB))
QUAD_COLORS <- setNames(c("#B8B8B8", "#2E7D32", X_HEX, "#6A1B9A"), QUAD_LEVELS)

# --- the four DE / GSEA contrasts, named off the tags ------------------------
CONTRASTS <- c(sprintf("%saxis_%spos", X_TAG, Y_TAG),   # axis effect within SnC+
               sprintf("%saxis_%sneg", X_TAG, Y_TAG),   # axis effect within SnC-
               sprintf("%saxis_%spos", Y_TAG, X_TAG),   # sen effect within axis+
               sprintf("%saxis_%sneg", Y_TAG, X_TAG))   # sen effect within axis-
CONTRAST_HEADERS <- c(
    sprintf("%s+%s+ vs %s+%s−", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s−%s+ vs %s−%s−", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s+%s+ vs %s−%s+", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s+%s− vs %s−%s−", Y_TAG, X_LAB, Y_TAG, X_LAB))

# --- outputs are namespaced by axis so runs never overwrite each other -------
AXIS_FIG_DIR <- file.path(PATHS$figures, AXIS)
AXIS_RES_DIR <- file.path(PATHS$results, AXIS)
dir.create(AXIS_FIG_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(AXIS_RES_DIR, recursive = TRUE, showWarnings = FALSE)

# --- state annotation (all five; NOT gated by AXIS) --------------------------
STATE_ORDER  <- c("Homeostatic", "ARM", "IRM", "Stress", "DAM_like")
STATE_COLORS <- c(Homeostatic = "#7F8C8D", ARM = "#E67E22", IRM = "#2980B9",
                  Stress = "#8E44AD", DAM_like = "#C0392B")

# --- DE model design (confirmed 2026-07-28; supersedes the leaner ~pop+grp2+Sex)
DE_DESIGN     <- "~ 0 + pop + grp2 + Sex + Mean_Log_Library_Depth_scaled + Cohort"
DE_COEF       <- "popTEST"
DE_MIN_CELLS  <- 10L

# --- GSEA (consumed by the python notebook via gsea_config.json) -------------
GSEA_DBS      <- c("Reactome_2022")
GSEA_FDR_SIG  <- 0.05
GSEA_N_COMMON <- 6L      # sig in >=3 contrasts, top N by mean NES
GSEA_N_UNIQUE <- 4L      # sig in exactly 1 contrast, top N by |NES|
GSEA_RIBO_STRIP <- FALSE # keep translational terms; see Part F Why



# §0.3 — AXIS-AWARE HELPERS  (one definition each — see note)
# =============================================================================
# In the source notebook fit_one was defined 13x, resolve_score_col 6x,
# z_score 5x, venn2 4x, and save_figure was REDEFINED at cells 317/341,
# shadowing the canonical version above. Everything lives here now so a
# later cell cannot silently shadow it.
# =============================================================================

# --- resolve a score column from candidates ---------------------------------
resolve_score_col <- function(md, candidates, what = "score") {
    hit <- candidates[candidates %in% colnames(md)]
    if (!length(hit)) stop(sprintf("no %s column found; tried: %s",
                                   what, paste(candidates, collapse = ", ")))
    hit[1]
}

z_score <- function(x) as.numeric(scale(x))

# --- build the SnC x AXIS quadrant column -----------------------------------
# Mean-split on z-scored values, exactly as the source (cell 54).
build_quadrants <- function(obj, axis = AXIS, verbose = TRUE) {
    md   <- obj@meta.data
    spec <- AXIS_REGISTRY[[axis]]
    sen  <- resolve_score_col(md, SEN_CANDIDATES, "senescence")
    xcol <- resolve_score_col(md, spec$cols, paste(axis, "score"))
    sz <- z_score(md[[sen]]); xz <- z_score(md[[xcol]])
    lab <- spec$lab
    q <- ifelse(sz >  0 & xz >  0, sprintf("Sen+ %s+", lab),
        ifelse(sz >  0 & xz <= 0, sprintf("Sen+ %s-", lab),
        ifelse(sz <= 0 & xz >  0, sprintf("Sen- %s+", lab),
                                  sprintf("Sen- %s-", lab))))
    q[is.na(sz) | is.na(xz)] <- NA
    obj[[paste0("quad4_", tolower(axis))]] <- q
    obj$sen_z <- sz
    obj$axis_z <- xz
    if (verbose) {
        cat(sprintf("  scores : sen=%s  %s=%s\n", sen, axis, xcol))
        print(table(q, useNA = "ifany"))
        cat(sprintf("  cor(sen_z, %s_z) = %.3f   <- independence check\n",
                    tolower(axis), cor(sz, xz, use = "complete.obs")))
    }
    obj
}

# --- PREFLIGHT: fail loudly before any model runs ---------------------------
preflight_axis <- function(obj, axis = AXIS, min_cells = 50L, donor_col = "Donor") {
    md <- obj@meta.data; ok <- TRUE
    say <- function(pass, msg) {
        cat(sprintf("  [%s] %s\n", if (pass) "OK  " else "FAIL", msg))
        if (!pass) ok <<- FALSE
    }
    cat(sprintf("\n── PREFLIGHT · axis = %s ──\n", axis))
    say(axis %in% names(AXIS_REGISTRY), sprintf("axis '%s' is registered", axis))
    spec <- AXIS_REGISTRY[[axis]]
    xhit <- spec$cols[spec$cols %in% colnames(md)]
    say(length(xhit) > 0, sprintf("score column present (%s)",
        if (length(xhit)) xhit[1] else paste(spec$cols, collapse = "/")))
    shit <- SEN_CANDIDATES[SEN_CANDIDATES %in% colnames(md)]
    say(length(shit) > 0, "senescence score column present")
    if (length(xhit) && length(shit)) {
        say(sd(md[[xhit[1]]], na.rm = TRUE) > 0, "axis score is non-constant")
        say(sd(md[[shit[1]]], na.rm = TRUE) > 0, "senescence score is non-constant")
    }
    qc <- paste0("quad4_", tolower(axis))
    if (qc %in% colnames(md)) {
        tb <- table(md[[qc]])
        say(length(tb) == 4, sprintf("all four quadrants populated (%d)", length(tb)))
        say(all(tb >= min_cells), sprintf("every quadrant >= %d cells (min %d)",
                                          min_cells, min(tb)))
        if (donor_col %in% colnames(md)) {
            nd <- tapply(md[[donor_col]], md[[qc]], function(z) length(unique(z)))
            say(all(nd >= 2), sprintf("every quadrant has >=2 donors (min %d)", min(nd)))
        }
    } else cat(sprintf("  [--  ] %s not built yet (run B1)\n", qc))
    cat(sprintf("── %s ──\n\n", if (ok) "PASS" else "STOP: fix before proceeding"))
    invisible(ok)
}

# --- ONE mixed-model fitter (replaces 13 copies of fit_one) -----------------
# formula_str is built by the caller, so every Part C analysis is this
# function with a different formula and a different subset.
fit_lmm <- function(df, formula_str, term, label = NA_character_) {
    fit <- tryCatch(lmerTest::lmer(as.formula(formula_str), data = df,
                                   REML = TRUE,
                                   control = lme4::lmerControl(
                                       optimizer = "bobyqa",
                                       optCtrl = list(maxfun = 2e5))),
                    error = function(e) NULL, warning = function(w) NULL)
    if (is.null(fit)) return(data.frame(label = label, term = term,
                                        beta = NA, se = NA, ci_low = NA,
                                        ci_high = NA, p_value = NA,
                                        n = nrow(df), converged = FALSE))
    co <- summary(fit)$coefficients
    if (!term %in% rownames(co)) return(data.frame(label = label, term = term,
                                        beta = NA, se = NA, ci_low = NA,
                                        ci_high = NA, p_value = NA,
                                        n = nrow(df), converged = FALSE))
    b <- co[term, "Estimate"]; s <- co[term, "Std. Error"]
    data.frame(label = label, term = term, beta = b, se = s,
               ci_low = b - 1.96 * s, ci_high = b + 1.96 * s,
               p_value = co[term, "Pr(>|t|)"], n = nrow(df), converged = TRUE)
}

# --- ONE forest renderer (replaces 7 near-copies) ---------------------------
forest_plot <- function(res, title = "", xlab = "beta (95% CI)",
                        facet = NULL, color = X_HEX) {
    stopifnot(all(c("label", "beta", "ci_low", "ci_high") %in% names(res)))
    if (!"p_adj" %in% names(res))
        res$p_adj <- p.adjust(res$p_value, method = STATISTICAL_PARAMS$fdr_method)
    res$sig <- sig_stars(res$p_adj)
    res$label <- factor(res$label, levels = rev(unique(res$label)))
    p <- ggplot(res, aes(x = beta, y = label)) +
        geom_vline(xintercept = 0, linetype = "dashed",
                   colour = "grey60", linewidth = 0.3) +
        geom_errorbarh(aes(xmin = ci_low, xmax = ci_high),
                       height = 0, linewidth = 0.4, colour = color) +
        geom_point(size = 1.8, colour = color) +
        geom_text(aes(x = ci_high, label = sig), hjust = -0.35,
                  size = 2.6, na.rm = TRUE) +
        labs(title = title, x = xlab, y = NULL) +
        theme_clean() +
        theme(panel.grid.major.y = element_line(colour = "grey92", linewidth = 0.25))
    if (!is.null(facet)) p <- p + facet_wrap(as.formula(paste("~", facet)), scales = "free_x")
    p + coord_cartesian(clip = "off")
}

# --- block banner: prints the Why with the axis resolved --------------------
say_block <- function(id, title, why = NULL) {
    cat("\n", strrep("═", 76), "\n", sep = "")
    cat(sprintf("%s  ·  %s\n", id, sprintf(title, X_LAB)))
    cat(strrep("═", 76), "\n", sep = "")
    if (!is.null(why)) cat(sprintf("WHY: %s\n\n", sprintf(why, X_LAB)))
}

# X_COL is resolved in the load section, once the object exists:
#     X_COL <- resolve_score_col(mg@meta.data, AXIS_SPEC$cols, 'axis score')
# Every downstream cell reads X_COL, never a literal score column name.

---
## 02 · Pseudobulk build and pairing QC

**Why.** Five steps, each with a check, because a silent failure here
propagates into every contrast below.

1. Quadrant labels and group mapping — sanity check the counts.
2. Donor-level cohort and group structure — decides whether cohorts can pool.
3. Contrast setup — population factor plus an explicit pairing check.
4. Aggregation, with the per-sample cell counts merged in **before** filtering
   so the filter can be seen to do what it claims.
5. Minimum-cell filter, then re-pair — dropping a thin sample can leave its
   donor unpaired, so pairing has to be recomputed after filtering, not before.

Then the design matrix, with the contrast verified rather than assumed.

In [ ]:
# AXIS: 2 hardcoded 'Score_DAM_like' reference(s) replaced with
#       X_COL, resolved from AXIS in the config cell. Setting
#       AXIS <- 'IRM' now actually analyses IRM.
# STEP 1 — quadrant labels + group mapping · sanity check counts
suppressPackageStartupMessages({ library(Seurat); library(dplyr) })
stopifnot(exists("obj_ct"))

md0 <- obj_ct@meta.data
stopifnot(all(c("senescence_score",X_COL,"Study_Group","Donor","Sex") %in% colnames(md0)))

# z-score the two axes (whole-population scaling, as before)
sen_z <- as.numeric(scale(md0$senescence_score))
dam_z <- as.numeric(scale(md0[[X_COL]]))

obj_ct$quad <- ifelse(sen_z>0  & dam_z>0,  "SnCpos_DAMpos",
               ifelse(sen_z>0  & dam_z<=0, "SnCpos_DAMneg",
               ifelse(sen_z<=0 & dam_z>0,  "SnCneg_DAMpos", "SnCneg_DAMneg")))

# disease group mapping
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")
obj_ct$grp2 <- unname(gmap[as.character(md0$Study_Group)])

cat("Study_Group values present:\n"); print(table(md0$Study_Group, useNA="ifany"))
cat("\ngrp2 mapping result:\n"); print(table(obj_ct$grp2, useNA="ifany"))
cat("\nCells per quadrant x group:\n")
print(table(obj_ct$quad, obj_ct$grp2, useNA="ifany"))
cat("\nCohort levels:\n"); print(table(obj_ct$Cohort, useNA="ifany"))
cat("\nSex levels:\n"); print(table(obj_ct$Sex, useNA="ifany"))
cat("\nDonors total:", length(unique(obj_ct$Donor)), "\n")

In [ ]:
# STEP 1b — donor-level cohort/group structure (decide cohort pooling)
suppressPackageStartupMessages({ library(dplyr) })

# restrict to the Old cells we'll actually use (drop Young)
md <- obj_ct@meta.data %>% filter(!is.na(grp2))
cat("After dropping Young (NA grp2):", nrow(md), "cells |",
    length(unique(md$Donor)), "donors\n\n")

# donors per cohort
cat("Donors per Cohort (Old AD + Old HC only):\n")
donor_cohort <- md %>% distinct(Donor, Cohort, grp2)
print(table(donor_cohort$Cohort))

cat("\nCohort x group (donor counts) — check for confounding:\n")
print(table(donor_cohort$Cohort, donor_cohort$grp2))

cat("\nSex x group (donor counts):\n")
print(table(distinct(md, Donor, Sex, grp2)$Sex, distinct(md, Donor, Sex, grp2)$grp2))

In [ ]:
# STEP 2 — single contrast setup (DAMaxis_SnCpos) · pop factor + pairing check
#   design will be ~ pop + grp2 + Sex (intercept; popTEST coef = TEST - REF)
suppressPackageStartupMessages({ library(Seurat); library(dplyr) })

TEST <- "SnCpos_DAMpos"; REF <- "SnCpos_DAMneg"   # first contrast to validate

OBJ <- obj_ct[, obj_ct$quad %in% c(TEST,REF) & !is.na(obj_ct$grp2)]
OBJ$pop <- factor(ifelse(OBJ$quad==TEST, "TEST", "REF"), levels=c("REF","TEST"))  # REF = baseline
if (!"nCount_RNA" %in% colnames(OBJ@meta.data))
  OBJ$nCount_RNA <- colSums(GetAssayData(OBJ, slot="counts"))

cat("Contrast:", TEST, "(TEST) vs", REF, "(REF)\n")
cat("Cells:", ncol(OBJ), "| donors:", length(unique(OBJ$Donor)), "\n\n")

cat("pop x grp2 (cell counts):\n"); print(table(OBJ$pop, OBJ$grp2))
cat("\nDonor pairing — donors with BOTH pops (cell counts per donor):\n")
tab <- table(OBJ$Donor, OBJ$pop)
both <- rownames(tab)[tab[,"TEST"]>0 & tab[,"REF"]>0]
cat("  donors with both quadrants:", length(both), "of", nrow(tab), "\n")
cat("  donors missing one side:", nrow(tab)-length(both), "(will be dropped)\n")
cat("\n  distribution of per-donor cell counts (both pops):\n")
print(summary(as.vector(tab[both,])))
cat("\n  donors with <10 cells in either pop (will be dropped at min-cell step):\n")
thin <- both[apply(tab[both,], 1, min) < 10]
cat("  ", length(thin), "donors\n")

In [ ]:
# STEP 3 — pseudobulk aggregation + n_cells merge (validate before filtering)
suppressPackageStartupMessages({ library(Seurat); library(dplyr) })

DONOR_COL<-"Donor"; SEX_COL<-"Sex"
gbc <- c(DONOR_COL, "grp2", "pop", SEX_COL)        # grouping for pseudobulk (no Cohort)
gbc <- gbc[gbc %in% colnames(OBJ@meta.data)]
cat("Pseudobulk grouping columns:", paste(gbc, collapse=", "), "\n\n")

# per-sample cell counts BEFORE aggregating
donor_covs <- OBJ@meta.data %>%
  group_by(across(all_of(gbc))) %>%
  summarise(n_cells = n(), .groups="drop") %>% as.data.frame()
donor_covs$.key <- apply(donor_covs[,gbc,drop=FALSE], 1, function(r)
    paste(gsub("_","-", as.character(r)), collapse="_"))
cat("Expected pseudobulk samples:", nrow(donor_covs), "\n")

# aggregate
pseudo <- AggregateExpression(OBJ, assays="RNA", return.seurat=TRUE, group.by=gbc)
cat("AggregateExpression produced:", ncol(pseudo), "samples\n")
cat("\nFirst few pseudobulk sample names (the keys Seurat built):\n")
print(head(colnames(pseudo), 4))
cat("\nFirst few of our constructed keys:\n")
print(head(donor_covs$.key, 4))

# merge n_cells
m <- match(colnames(pseudo), donor_covs$.key)
if (any(is.na(m))) {
  cat("\n⚠ direct key match missed", sum(is.na(m)), "— trying rebuild from pseudo meta\n")
  pm <- pseudo@meta.data[, gbc, drop=FALSE]
  pkey <- apply(pm, 1, function(r) paste(gsub("_","-",as.character(r)), collapse="_"))
  m <- match(pkey, donor_covs$.key)
}
pseudo$n_cells <- donor_covs$n_cells[m]
cat(sprintf("\nmatched n_cells: %d / %d samples\n", sum(!is.na(pseudo$n_cells)), ncol(pseudo)))
cat("n_cells summary:\n"); print(summary(pseudo$n_cells))

In [ ]:
# STEP 4 — min-cell filter + re-pair (drop thin samples & unpaired donors)
MIN_CELLS <- 10

nc <- pseudo$n_cells
cat("Before filter:", ncol(pseudo), "samples\n")
pseudo <- pseudo[, !is.na(nc) & nc >= MIN_CELLS]
cat("After >=", MIN_CELLS, "cells:", ncol(pseudo), "samples\n")

pseudo$pop <- factor(pseudo$pop, levels=c("REF","TEST"))
pd <- pseudo$Donor; pp <- pseudo$pop
tab <- table(pd, pp)
paired <- rownames(tab)[tab[,"TEST"]>0 & tab[,"REF"]>0]
pseudo <- pseudo[, pd %in% paired]
cat("After re-pairing (donor has both pops):", ncol(pseudo), "samples |", length(paired), "donors\n\n")

cat("Final pop x grp2 (sample counts):\n"); print(table(pseudo$pop, pseudo$grp2))
cat("\nFinal Sex x pop:\n"); print(table(pseudo$Sex, pseudo$pop))

In [ ]:
# STEP 5 — DESIGN MATRIX (~ pop + grp2 + Sex, WITH intercept) + verify contrast
#   popTEST coefficient = TEST - REF (because intercept present + REF is baseline)
meta <- pseudo@meta.data
meta$pop  <- factor(meta$pop,  levels=c("REF","TEST"))   # REF = baseline
meta$grp2 <- factor(meta$grp2)
meta$Sex  <- factor(meta$Sex)

# intercept design — NOT 0+ ; popTEST is already the difference
design <- model.matrix(~ pop + grp2 + Sex, data=meta)

cat("Design columns:\n"); print(colnames(design))
cat("\nDesign rank:", qr(design)$rank, "of", ncol(design), "columns",
    ifelse(qr(design)$rank==ncol(design), "(full rank ✓)", "(RANK DEFICIENT ✗)"), "\n")
cat("\nColumn sums (how many samples load on each):\n"); print(colSums(design))
cat("\nHead of design:\n"); print(head(design, 4))
cat("\n>>> The coefficient we will test is 'popTEST'.\n")
cat(">>> With the intercept present and REF as baseline,\n")
cat(">>> popTEST = (mean expression in TEST) - (mean expression in REF) = the contrast we want.\n")
cat(">>> This is NOT the 0+ trap (which tested absolute expression). \n")

---
## 03 · Four-contrast limma-voom loop

**Why.** One loop, four contrasts, identical machinery — so a difference
between contrasts is a difference in biology and not in how they were fit.

**Test.** limma-voom with `duplicateCorrelation` blocking on donor, which
handles the donors contributing to both arms of a contrast.

**Formula.** `~ 0 + pop + grp2 + Sex + Mean_Log_Library_Depth_scaled + Cohort`,
testing `popTEST`.

**Display.** Per-contrast fit summary and DEG counts at the configured FDR.

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['IRM'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# 2×2 QUADRANT DE — all four axis contrasts (limma-voom, donor-paired)
#   axis1 = Y (senescence), axis2 = X (the activation axis: DAM or IRM)
#   A: X axis | Y+   (Y+X+ vs Y+X-)   B: X axis | Y-   (Y-X+ vs Y-X-)
#   C: Y axis | X+   (Y+X+ vs Y-X+)   D: Y axis | X-   (Y+X- vs Y-X-)
suppressPackageStartupMessages({ library(Seurat); library(dplyr); library(edgeR); library(limma) })

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
OBJ0   <- obj_ct
Y_COL  <- "senescence_score"; Y_TAG <- "SnC"   # axis 1 (rows of the quadrant)
X_COL  <- AXIS_SPEC$cols[1];        X_TAG <- X_TAG   # axis 2 — swap Score_DAM_like / "DAM" for the DAM run
GROUPS <- c("Old_AD"="AD", "Old_Healthy_Control"="Control")
DONOR_COL<-"Donor"; SEX_COL<-"Sex"; AGE_COL<-"Age"; COHORT_COL<-"Cohort"
HAS_COHORT <- TRUE; MIN_CELLS <- 10; FDR_THRESHOLD <- 0.05; MIN_PAIRED <- 10

qq <- function(yhi, xhi) sprintf("%s%s_%s%s", Y_TAG, ifelse(yhi,"pos","neg"),
                                              X_TAG, ifelse(xhi,"pos","neg"))
YP_XP<-qq(T,T); YP_XN<-qq(T,F); YN_XP<-qq(F,T); YN_XN<-qq(F,F)

md <- OBJ0@meta.data
stopifnot(Y_COL %in% colnames(md), X_COL %in% colnames(md))
y_z <- as.numeric(scale(md[[Y_COL]])); x_z <- as.numeric(scale(md[[X_COL]]))
OBJ0$quad <- ifelse(y_z>0 & x_z>0, YP_XP,
             ifelse(y_z>0 & x_z<=0, YP_XN,
             ifelse(y_z<=0 & x_z>0, YN_XP, YN_XN)))
OBJ0$grp2 <- unname(GROUPS[as.character(md$Study_Group)])
OBJ0 <- OBJ0[, !is.na(OBJ0$grp2)]
cat("Cells per quadrant \u00d7 group:\n"); print(table(OBJ0$quad, OBJ0$grp2))

# four contrasts derived from the tags (test vs ref; logFC>0 = up in test)
CONTRASTS <- list(
  A = list(test=YP_XP, ref=YP_XN, tag=sprintf("%saxis_%spos", X_TAG, Y_TAG)),  # X axis | Y+
  B = list(test=YN_XP, ref=YN_XN, tag=sprintf("%saxis_%sneg", X_TAG, Y_TAG)),  # X axis | Y-
  C = list(test=YP_XP, ref=YN_XP, tag=sprintf("%saxis_%spos", Y_TAG, X_TAG)),  # Y axis | X+
  D = list(test=YP_XN, ref=YN_XN, tag=sprintf("%saxis_%sneg", Y_TAG, X_TAG)))  # Y axis | X-

run_contrast <- function(OBJ0, test, ref, tag) {
  cat("\n", strrep("=",70), "\n", tag, "  (", test, " vs ", ref, ")\n", strrep("=",70), "\n", sep="")
  OBJ <- OBJ0
  OBJ$popX <- ifelse(OBJ$quad==test, "TEST", ifelse(OBJ$quad==ref, "REF", NA))
  OBJ <- OBJ[, !is.na(OBJ$popX)]
  gbc <- c(DONOR_COL,"grp2","popX",SEX_COL,COHORT_COL)
  if (!"nCount_RNA" %in% colnames(OBJ@meta.data))
    OBJ$nCount_RNA <- colSums(GetAssayData(OBJ, layer="counts"))
  donor_covs <- OBJ@meta.data %>% group_by(across(all_of(gbc))) %>%
    summarise(Mean_Age=mean(as.numeric(.data[[AGE_COL]]),na.rm=TRUE),
              Mean_Log_Library_Depth=mean(log10(nCount_RNA+1),na.rm=TRUE),
              n_cells=n(), .groups="drop") %>% as.data.frame()
  pseudo <- AggregateExpression(OBJ, assays="RNA", return.seurat=TRUE, group.by=gbc)

  pb_donor <- pseudo[[DONOR_COL,drop=TRUE]]; tb <- table(pb_donor, pseudo[["popX",drop=TRUE]])
  if (!all(c("TEST","REF") %in% colnames(tb))) { cat("  \u2717 a quadrant is absent \u2014 skipping\n"); return(NULL) }
  paired <- rownames(tb)[tb[,"TEST"]>0 & tb[,"REF"]>0]
  pseudo <- pseudo[, pb_donor %in% paired]
  for (col in gbc) if (is.character(donor_covs[[col]])||is.factor(donor_covs[[col]]))
    donor_covs[[col]] <- gsub("_","-",as.character(donor_covs[[col]]))
  pm <- pseudo@meta.data; pm$row_id <- rownames(pm); for (col in gbc) pm[[col]] <- as.character(pm[[col]])
  mgm <- merge(pm, donor_covs, by=gbc, all.x=TRUE, sort=FALSE); mgm <- mgm[match(pm$row_id, mgm$row_id),]
  pseudo[["Mean_Log_Library_Depth"]]<-mgm$Mean_Log_Library_Depth; pseudo[["n_cells"]]<-mgm$n_cells

  pseudo <- pseudo[, pseudo[["n_cells",drop=TRUE]] >= MIN_CELLS]
  pb_donor <- pseudo[[DONOR_COL,drop=TRUE]]; tb <- table(pb_donor, pseudo[["popX",drop=TRUE]])
  if (!all(c("TEST","REF") %in% colnames(tb))) { cat("  \u2717 no paired donors after filter \u2014 skipping\n"); return(NULL) }
  paired <- rownames(tb)[tb[,"TEST"]>0 & tb[,"REF"]>0]; pseudo <- pseudo[, pb_donor %in% paired]
  cat(sprintf("  paired donors: %d | samples: %d\n", length(paired), ncol(pseudo)))
  if (length(paired) < MIN_PAIRED) cat(sprintf("  \u26a0 LOW POWER: only %d paired donors\n", length(paired)))

  counts_mat <- as.matrix(GetAssayData(pseudo, layer="counts")); meta_df <- pseudo@meta.data
  meta_df[[SEX_COL]]<-factor(meta_df[[SEX_COL]]); meta_df[[DONOR_COL]]<-factor(meta_df[[DONOR_COL]])
  if (HAS_COHORT){ meta_df[[COHORT_COL]]<-factor(meta_df[[COHORT_COL]]); levels(meta_df[[COHORT_COL]])<-make.names(levels(meta_df[[COHORT_COL]])) }
  meta_df$pop  <- factor(ifelse(meta_df$popX=="TEST","TEST","REF"), levels=c("REF","TEST"))
  meta_df$grp2 <- factor(meta_df$grp2)
  meta_df$Mean_Log_Library_Depth_scaled <- scale(meta_df$Mean_Log_Library_Depth)[,1]
  dge <- DGEList(counts=counts_mat, samples=meta_df, group=meta_df$pop)
  keep <- filterByExpr(dge, group=meta_df$pop); dge <- dge[keep,,keep.lib.sizes=FALSE]
  dge <- calcNormFactors(dge, method="TMM")

  # drop covariates with a single level (rare quadrants may lack both groups/sexes/cohorts)
  rhs <- c("0+pop","grp2","Sex","Mean_Log_Library_Depth_scaled", if (HAS_COHORT) "Cohort")
  for (v0 in intersect(c("grp2","Sex","Cohort"), rhs))
    if (nlevels(droplevels(meta_df[[v0]]))<2) { rhs<-setdiff(rhs,v0); cat(sprintf("  (dropped %s: single level)\n",v0)) }
  design <- model.matrix(as.formula(paste("~",paste(rhs,collapse="+"))), data=meta_df)
  colnames(design) <- make.names(colnames(design))
  if (qr(design)$rank < ncol(design)) { cat("  \u2717 rank-deficient design \u2014 skipping\n"); return(NULL) }
  v <- voom(dge, design, plot=FALSE)
  corfit <- duplicateCorrelation(v, design, block=meta_df[[DONOR_COL]])
  cat(sprintf("  donor consensus corr: %.3f | genes: %d\n", corfit$consensus.correlation, nrow(dge)))
  fit <- lmFit(v, design, block=meta_df[[DONOR_COL]], correlation=corfit$consensus.correlation)
  fit <- contrasts.fit(fit, makeContrasts(popTEST - popREF, levels=design)); fit <- eBayes(fit)
  res <- topTable(fit, coef=1, number=Inf, sort.by="P"); res$gene <- rownames(res)
  res$direction <- ifelse(res$adj.P.Val<FDR_THRESHOLD & res$logFC>0,"Up_TEST",
                   ifelse(res$adj.P.Val<FDR_THRESHOLD & res$logFC<0,"Up_REF","NS"))
  cat(sprintf("  DEGs FDR<%.2f: %d up in %s, %d up in %s\n",
              FDR_THRESHOLD, sum(res$direction=="Up_TEST"), test, sum(res$direction=="Up_REF"), ref))
  save_table(res, sprintf("pseudobulk_DE_%s_microglia", tag))
  res
}

all_res <- lapply(CONTRASTS, function(cc) run_contrast(OBJ0, cc$test, cc$ref, cc$tag))
names(all_res) <- sapply(CONTRASTS, `[[`, "tag")
cat("\n", strrep("=",70), "\nSUMMARY  (Y=", Y_TAG, ", X=", X_TAG, ")\n", strrep("=",70), "\n", sep="")
for (nm in names(all_res)) {
  r <- all_res[[nm]]
  if (is.null(r)) { cat(sprintf("  %-20s : skipped (insufficient data)\n", nm)); next }
  cat(sprintf("  %-20s : %d up TEST / %d up REF (of %d)\n", nm,
              sum(r$direction=="Up_TEST"), sum(r$direction=="Up_REF"), nrow(r)))
}
cat("\n\u2713 all contrasts done \u2014 results in all_res list.\n")

---
## 04 · Volcano panel

**Why.** Four volcanoes on shared axes. Shared limits matter — otherwise a
contrast with a weak effect is rescaled into looking like a strong one.

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['IRM'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# 2×2 VOLCANO PANEL — four quadrant axis contrasts (Sen × IRM)
#   RED = up in TEST (left term) · BLUE = up in REF (right term)
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(ggrepel); library(patchwork) })

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
Y_TAG <- "SnC"; X_TAG <- X_TAG            # must match the DE run that filled all_res
FDR_CUT <- 0.05; LFC_CUT <- 0.25; N_LABEL <- 10
UP_COL <- "#C0392B"; DN_COL <- "#2471A3"
FILE_TAG <- "SnC_IRM"
# panel keys must equal names(all_res); tags from the DE cell:
K_XpY <- sprintf("%saxis_%spos", X_TAG, Y_TAG)   # X axis | Y+   (clean)
K_XnY <- sprintf("%saxis_%sneg", X_TAG, Y_TAG)   # X axis | Y-   (clean)
K_YpX <- sprintf("%saxis_%spos", Y_TAG, X_TAG)   # Y axis | X+   (circular)
K_YnX <- sprintf("%saxis_%sneg", Y_TAG, X_TAG)   # Y axis | X-   (circular)

yp<-"SnC+"; yn<-"SnC\u2212"; xp<-paste0(X_TAG,"+"); xn<-paste0(X_TAG,"\u2212")
PANELS <- setNames(list(
  list(t=sprintf("%s axis | senescent\n%s%s vs %s%s", X_TAG, yp,xp, yp,xn),        up=paste0(yp,xp), dn=paste0(yp,xn)),
  list(t=sprintf("%s axis | non-senescent\n%s%s vs %s%s", X_TAG, yn,xp, yn,xn),    up=paste0(yn,xp), dn=paste0(yn,xn)),
  list(t=sprintf("Sen axis | %s+ \n%s%s vs %s%s", X_TAG, yp,xp, yn,xp), up=paste0(yp,xp), dn=paste0(yn,xp)),
  list(t=sprintf("Sen axis | %s\u2212 \n%s%s vs %s%s", X_TAG, yp,xn, yn,xn), up=paste0(yp,xn), dn=paste0(yn,xn))),
  c(K_XpY, K_XnY, K_YpX, K_YnX))

one_volcano <- function(res, cfg) {
  if (is.null(res)) return(patchwork::plot_spacer() +
      labs(title=paste0(cfg$t, "\n(skipped — insufficient data)")) +
      theme_void() + theme(plot.title=element_text(size=7, hjust=0)))
  d <- res %>% mutate(
    neglog10 = -log10(adj.P.Val),
    cls = case_when(adj.P.Val<FDR_CUT & logFC>= LFC_CUT ~ "UP",
                    adj.P.Val<FDR_CUT & logFC<=-LFC_CUT ~ "DN", TRUE ~ "NS"))
  YCAP <- quantile(d$neglog10[is.finite(d$neglog10)], 0.999)
  d$y <- pmin(d$neglog10, YCAP); d$capped <- d$neglog10 > YCAP
  xlim <- max(abs(d$logFC))*1.05
  lab <- bind_rows(
    d %>% filter(cls=="UP") %>% arrange(adj.P.Val) %>% head(N_LABEL),
    d %>% filter(cls=="DN") %>% arrange(adj.P.Val) %>% head(N_LABEL))
  pal <- c(UP=UP_COL, DN=DN_COL, NS="grey80")
  nU <- sum(d$cls=="UP"); nD <- sum(d$cls=="DN")
  ggplot(d, aes(logFC, y)) +
    geom_vline(xintercept=c(-LFC_CUT,LFC_CUT), linetype="dashed", colour="grey75", linewidth=0.25) +
    geom_hline(yintercept=-log10(FDR_CUT), linetype="dashed", colour="grey75", linewidth=0.25) +
    geom_point(data=subset(d,cls=="NS"), colour="grey80", size=0.5, alpha=0.3, shape=16) +
    geom_point(data=subset(d,cls!="NS"), aes(colour=cls), size=0.9, alpha=0.8, shape=16) +
    geom_point(data=subset(d,capped & cls!="NS"), aes(colour=cls), y=YCAP, shape=2, size=1.2, stroke=0.4) +
    geom_text_repel(data=lab, aes(label=gene), colour="black", size=2.2, fontface="italic",
                    segment.size=0.2, segment.color="grey60", min.segment.length=0,
                    box.padding=0.28, max.overlaps=Inf) +
    annotate("text", x= xlim*0.95, y=0, hjust=1, vjust=0, size=2.4, colour=UP_COL,
             label=sprintf("%d \u2191 %s", nU, cfg$up)) +
    annotate("text", x=-xlim*0.95, y=0, hjust=0, vjust=0, size=2.4, colour=DN_COL,
             label=sprintf("%d \u2191 %s", nD, cfg$dn)) +
    scale_colour_manual(values=pal, guide="none") +
    scale_x_continuous(limits=c(-xlim,xlim)) +
    labs(x=expression(log[2]~FC), y=expression(-log[10]~FDR), title=cfg$t) +
    theme_classic(base_size=8) +
    theme(plot.title=element_text(face="bold", size=8, hjust=0, lineheight=1.05),
          axis.text=element_text(size=7, colour="black"),
          panel.border=element_rect(colour="black", fill=NA, linewidth=0.5),
          axis.line=element_blank(), axis.ticks=element_line(linewidth=0.35),
          plot.margin=margin(4,6,4,4))
}

missing <- setdiff(names(PANELS), names(all_res))
if (length(missing)) cat("\u26a0 all_res missing:", paste(missing, collapse=", "), "\n")
plots <- lapply(names(PANELS), function(nm) one_volcano(all_res[[nm]], PANELS[[nm]]))
combined <- (plots[[1]] | plots[[2]]) / (plots[[3]] | plots[[4]]) +
  plot_annotation(title=sprintf("Senescence \u00d7 %s: axis-resolved transcriptional programs (microglia)", X_TAG),
                  subtitle="Red = up in left term \u00b7 Blue = up in right term \u00b7 bottom row = senescence axis (defining genes expected)",
                  theme=theme(plot.title=element_text(face="bold", size=11),
                              plot.subtitle=element_text(size=8, color="grey40")))
options(repr.plot.width=10, repr.plot.height=9); print(combined)
save_figure(combined, sprintf("volcano_2x2_quadrant_axes_%s_microglia", FILE_TAG), width=10, height=9)
cat("\n\u2713 2x2 volcano panel saved\n")

---
## 05 · DEG counts

**Why.** The four contrasts side by side as diverging bars, up to the right and
down to the left. This is where an asymmetry between the axis contrasts and the
senescence contrasts becomes visible.

In [ ]:
# DEG COUNT — diverging bars (compact): up → right (red), down → left (blue)
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(tidyr); library(stringr) })

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
Y_TAG <- "SnC"; X_TAG <- X_TAG          # must match the DE run that filled all_res
FDR_CUT <- 0.05; LFC_CUT <- 0.25
UP_COL <- "#C0392B"; DN_COL <- "#2471A3"
FILE_TAG <- "SnC_IRM"
yp<-paste0(Y_TAG,"+"); yn<-paste0(Y_TAG,"-"); xp<-paste0(X_TAG,"+"); xn<-paste0(X_TAG,"-")
META <- tibble::tribble(
  ~key,                                   ~axis,                          ~label,
  sprintf("%saxis_%spos",X_TAG,Y_TAG), sprintf("%s axis",X_TAG),  sprintf("%s%s vs %s%s", yp,xp, yp,xn),  # clean
  sprintf("%saxis_%sneg",X_TAG,Y_TAG), sprintf("%s axis",X_TAG),  sprintf("%s%s vs %s%s", yn,xp, yn,xn),  # clean
  sprintf("%saxis_%spos",Y_TAG,X_TAG), "Sen axis",     sprintf("%s%s vs %s%s", yp,xp, yn,xp),  # circular
  sprintf("%saxis_%sneg",Y_TAG,X_TAG), "Sen axis",     sprintf("%s%s vs %s%s", yp,xn, yn,xn))  # circular

miss <- setdiff(META$key, names(all_res))
if (length(miss)) cat("\u26a0 all_res missing:", paste(miss, collapse=", "), "\n")

cnt <- lapply(META$key, function(k){
  r <- all_res[[k]]
  if (is.null(r)) return(data.frame(key=k, Up=NA_integer_, Down=NA_integer_))
  data.frame(key=k,
             Up   =  sum(r$adj.P.Val<FDR_CUT & r$logFC>= LFC_CUT),
             Down = -sum(r$adj.P.Val<FDR_CUT & r$logFC<=-LFC_CUT))
}) %>% bind_rows() %>% left_join(META, by="key") %>%
  pivot_longer(c(Up,Down), names_to="dir", values_to="n")

cnt$label <- str_wrap(cnt$label, width=12)
cnt$label <- factor(cnt$label, levels=str_wrap(rev(META$label), width=12))
cnt$dir   <- factor(cnt$dir, levels=c("Up","Down"))
cnt$axis  <- factor(cnt$axis, levels=unique(META$axis))

rng <- range(c(cnt$n, 0), na.rm=TRUE)
xlo <- rng[1]*1.12; xhi <- rng[2]*1.12

p <- ggplot(cnt, aes(x=n, y=label, fill=dir)) +
  geom_vline(xintercept=0, colour="grey40", linewidth=0.4) +
  geom_col(width=0.6, colour="black", linewidth=0.25) +
  geom_text(aes(label=abs(n), hjust=ifelse(n>=0, -0.25, 1.25)),
            size=2.9, fontface="bold", na.rm=TRUE) +
  facet_grid(axis~., scales="free_y", space="free_y") +
  scale_fill_manual(values=c(Up=UP_COL, Down=DN_COL), name=NULL,
                    labels=c("up in left term", "up in right term")) +
  scale_x_continuous(limits=c(xlo, xhi), labels=function(x) abs(x)) +
  labs(x="# of DEGs", y=NULL,
       title=sprintf("DEG counts: senescence \u00d7 %s axis contrasts (microglia)", X_TAG)) +
  theme_classic(base_size=10) +
  theme(plot.title=element_text(face="bold", size=10.5),
        axis.text.y=element_text(size=8, lineheight=0.9),
        strip.background=element_blank(), strip.text=element_text(face="bold", size=9.5),
        panel.border=element_rect(colour="black", fill=NA, linewidth=0.5),
        axis.line=element_blank(), legend.position="top",
        plot.margin=margin(4,6,4,4))
options(repr.plot.width=6.5, repr.plot.height=4); print(p)
save_figure(p, sprintf("DEG_counts_diverging_2x2_axes_%s_microglia", FILE_TAG), width=6.5, height=4)
cat("\n\u2713 compact diverging bar plot saved\n")

---
## 06 · Venns — consistency and distinctness

**Why.** Two questions. *Within* an axis, do the SnC+ and SnC− contrasts recover
the same genes — is the axis effect stable across senescence strata? *Between*
axes, are the senescence genes and the activation genes distinct sets?

Large within-axis overlap plus small between-axis overlap is the separability
result at gene level. That is the claim module 10 makes with scores, tested here
with genes.

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['IRM'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# VENN (pure ggplot) — labels placed OUTSIDE each circle, no overlap
suppressPackageStartupMessages({ library(dplyr); library(ggplot2); library(patchwork) })

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
Y_TAG <- "SnC"; X_TAG <- X_TAG          # must match the DE run that filled all_res
Y_NAME <- "senescence"                  # spelled-out axis names for titles/labels
X_NAME <- "IRM"
FDR_CUT <- 0.05; LFC_CUT <- 0.25
FILL_A <- "#6BAED6"; FILL_B <- "#FB9A99"
FILE_TAG <- "SnC_IRM"
FIG_DIR <- file.path(Sys.getenv("SENESCENCE_DATA"), "brain/module_06_dge/figures")
dir.create(FIG_DIR, recursive=TRUE, showWarnings=FALSE)
if (!exists("save_figure")) {
  save_figure <- function(plot,name,width=6,height=5,dpi=300){
    for (fmt in c("png","pdf","svg"))
      ggsave(file.path(FIG_DIR,paste0(name,".",fmt)), plot, width=width, height=height,
             dpi=dpi, bg="white", device=if(fmt=="pdf") cairo_pdf else fmt)
    cat(sprintf("  saved: %s.{png,pdf,svg}\n", name)) }
}
# contrast keys (match the DE cell tags)
K_XpY <- sprintf("%saxis_%spos", X_TAG, Y_TAG)   # IRM axis | SnC+   clean
K_XnY <- sprintf("%saxis_%sneg", X_TAG, Y_TAG)   # IRM axis | SnC-   clean
K_YpX <- sprintf("%saxis_%spos", Y_TAG, X_TAG)   # Sen axis | IRM+   circular
K_YnX <- sprintf("%saxis_%sneg", Y_TAG, X_TAG)   # Sen axis | IRM-   circular

degs <- function(key, dir=c("all","up","down")){
  dir<-match.arg(dir); r<-all_res[[key]]
  if (is.null(r)) return(character(0))
  if(dir=="up")   return(r$gene[r$adj.P.Val<FDR_CUT & r$logFC>= LFC_CUT])
  if(dir=="down") return(r$gene[r$adj.P.Val<FDR_CUT & r$logFC<=-LFC_CUT])
  r$gene[r$adj.P.Val<FDR_CUT & abs(r$logFC)>=LFC_CUT]
}
circle <- function(cx,cy,r,n=200){ t<-seq(0,2*pi,length.out=n); data.frame(x=cx+r*cos(t), y=cy+r*sin(t)) }
venn2 <- function(a, b, labA, labB, title){
  only_a<-length(setdiff(a,b)); only_b<-length(setdiff(b,a)); ov<-length(intersect(a,b))
  jac <- ov / max(1, length(union(a,b)))
  R<-0.85; cxL<--0.5; cxR<-0.5
  cL<-circle(cxL,0,R); cL$set<-"A"; cR<-circle(cxR,0,R); cR$set<-"B"
  ggplot() +
    geom_polygon(data=rbind(cL,cR), aes(x,y,group=set,fill=set), colour="black", linewidth=0.5, alpha=0.6) +
    scale_fill_manual(values=c(A=FILL_A,B=FILL_B), guide="none") +
    annotate("text", x=cxL-0.40, y=0, label=only_a, fontface="bold", size=4) +
    annotate("text", x=0,        y=0, label=ov,     fontface="bold", size=4) +
    annotate("text", x=cxR+0.40, y=0, label=only_b, fontface="bold", size=4) +
    annotate("text", x=-2.05, y=0, label=labA, fontface="bold", size=2.8, lineheight=0.9, hjust=0, colour="#2C5985") +
    annotate("text", x= 2.05, y=0, label=labB, fontface="bold", size=2.8, lineheight=0.9, hjust=1, colour="#9E3B3B") +
    annotate("text", x=0, y=-R-0.35, label=sprintf("Jaccard = %.2f", jac), size=3.2, colour="grey30") +
    coord_fixed(xlim=c(-2.1,2.1), ylim=c(-1.5,1.2), clip="off") +
    labs(title=title) + theme_void() +
    theme(plot.title=element_text(face="bold", size=10.5, hjust=0.5, lineheight=0.95),
          plot.margin=margin(6,4,6,4))
}
yp<-paste0(Y_TAG,"+"); yn<-paste0(Y_TAG,"-"); xp<-paste0(X_TAG,"+"); xn<-paste0(X_TAG,"-")

# v1: IRM program consistency — SnC+ vs SnC-   (both CLEAN)
v1 <- venn2(degs(K_XpY), degs(K_XnY),
            sprintf("%s axis\nin %s\n(%s%s vs\n%s%s)", X_NAME, yp, yp,xp, yp,xn),
            sprintf("%s axis\nin %s\n(%s%s vs\n%s%s)", X_NAME, yn, yn,xp, yn,xn),
            sprintf("%s program:\nsenescent vs non-senescent", X_NAME))
# v2: senescence program consistency — IRM+ vs IRM-   (both CIRCULAR)
v2 <- venn2(degs(K_YpX), degs(K_YnX),
            sprintf("%s axis\nin %s\n(%s%s vs\n%s%s)", Y_NAME, xp, yp,xp, yn,xp),
            sprintf("%s axis\nin %s\n(%s%s vs\n%s%s)", Y_NAME, xn, yp,xn, yn,xn),
            sprintf("%s program:\n%s vs %s", Y_NAME, xp, xn))
# v3: between axes — clean IRM axis vs circular senescence axis, in the SnC+IRM+ corner
v3 <- venn2(degs(K_XpY), degs(K_YpX),
            sprintf("%s axis\n(%s%s vs\n%s%s)", X_NAME, yp,xp, yp,xn),
            sprintf("%s axis \n(%s%s vs\n%s%s)", Y_NAME, yp,xp, yn,xp),
            sprintf("Between axes:\n%s vs %s", X_NAME, Y_NAME))

panel <- (v1 | v2 | v3) +
  plot_annotation(title=sprintf("Within-axis consistency vs between-axis distinctness (microglia DEGs, %s)", X_NAME),
                  theme=theme(plot.title=element_text(face="bold", size=12, hjust=0)))
options(repr.plot.width=13, repr.plot.height=4.5); print(panel)
save_figure(panel, sprintf("venn_axis_overlaps_pooled_%s", FILE_TAG), width=13, height=4.5)
cat("\n\u2713 done\n")

---
## 07 · Genome-maintenance heatmap

**Why.** Focused readout on the gene family that carries the mechanistic claim.
Log fold change across the senescence axis and the activation axis, side by side.

**Display.** Genes × two axes, so a gene that moves on one and not the other
reads immediately.

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['SenHi', 'SenLo'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# VOLCANO (R house style) — senescence axis, genome-maintenance genes highlighted
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(ggrepel) })
FDR_THRESHOLD <- 0.05

BASE <- file.path(Sys.getenv("SENESCENCE_DATA"), "brain/module_05_senescence_enrichment/disease/AD/psychad_ad/Microglia/results")
A <- read.csv(file.path(BASE, "pseudobulk_DE_SenHiDAMlo_vs_SenLoDAMlo_microglia.csv"), stringsAsFactors=FALSE)

# genome-maintenance highlight sets (among the senescence-unique genes)
CHROM <- c("AEBP2","DNMT1","KAT2B","KDM1B","KMT2E","RBBP4","SF3B1","SUZ12","TRIM24","NUCKS1","MED13","MED13L","MED4","NIPBL","RBM17")
DDR   <- c("FANCL","NASP","PRIMPOL","PRKDC","RAD54L2","SHLD2","XPA","CENPP","NEK9")
AUTO  <- c("ATG12","WIPI1","NPC1","GGA2","ATP6V1H","PIP4P2","SPPL3")

A$cat <- ifelse(A$gene %in% CHROM, "Chromatin/epigenetic",
         ifelse(A$gene %in% DDR,   "DNA repair",
         ifelse(A$gene %in% AUTO,  "Autophagy/lysosome", "other")))
A$cat <- factor(A$cat, levels=c("Chromatin/epigenetic","DNA repair","Autophagy/lysosome","other"))
A$nlfdr <- -log10(pmax(A$adj.P.Val, 1e-300))

# label highlighted genes that pass FDR<0.10
A$label <- ""
A$label[A$cat!="other" & A$adj.P.Val<0.10] <- A$gene[A$cat!="other" & A$adj.P.Val<0.10]

CATCOL <- c("Chromatin/epigenetic"="#6A3D9A", "DNA repair"="#E41A1C",
            "Autophagy/lysosome"="#FF7F00", "other"="grey75")

p <- ggplot(A, aes(logFC, nlfdr)) +
    # background (non-highlighted): NS pale, sig darker grey
    geom_point(data=subset(A, cat=="other" & adj.P.Val>=FDR_THRESHOLD),
               color="grey85", size=0.8, alpha=0.5) +
    geom_point(data=subset(A, cat=="other" & adj.P.Val<FDR_THRESHOLD),
               color="grey55", size=1.0, alpha=0.6) +
    # highlighted genes on top
    geom_point(data=subset(A, cat!="other"),
               aes(color=cat), size=2.6, alpha=0.9) +
    geom_vline(xintercept=0, linetype="dashed", color="grey40") +
    geom_hline(yintercept=-log10(FDR_THRESHOLD), linetype="dashed", color="grey40") +
    geom_text_repel(aes(label=label, color=cat), size=2.5, max.overlaps=20,
                    fontface="italic", segment.size=0.3, show.legend=FALSE) +
    scale_color_manual(values=CATCOL, name=NULL,
                       breaks=c("Chromatin/epigenetic","DNA repair","Autophagy/lysosome")) +
    labs(title="Senescence axis — genome-maintenance genes",
         subtitle="SenHi_DAMlo vs SenLo_DAMlo",
         x=expression(log[2]~"Fold Change (SenHi - SenLo)"),
         y=expression(-log[10]~"(FDR)")) +
    theme_classic(base_size=10) +
    theme(legend.position="right",
          plot.title=element_text(size=10, face="bold"),
          plot.subtitle=element_text(size=8, color="grey40"),
          panel.border=element_rect(color="black", fill=NA, linewidth=0.5))

options(repr.plot.width=7, repr.plot.height=6); print(p)
save_figure(p, "volcano_senescence_axis_genome_maintenance_microglia", width=7, height=6)
cat("\n✓ R-style volcano done.\n")

In [ ]:
# HEATMAP — genome-maintenance genes × {senescence axis, DAM axis} logFC
#   no .qs reload — uses the two DE CSVs
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(tidyr) })
BASE <- file.path(Sys.getenv("SENESCENCE_DATA"), "brain/module_05_senescence_enrichment/disease/AD/psychad_ad/Microglia/results")
A <- read.csv(file.path(BASE,"pseudobulk_DE_SenHiDAMlo_vs_SenLoDAMlo_microglia.csv"), stringsAsFactors=FALSE)  # senescence axis
B <- read.csv(file.path(BASE,"pseudobulk_DE_SenHiDAMhi_vs_SenHiDAMlo_microglia.csv"), stringsAsFactors=FALSE)  # DAM axis

# genome-maintenance gene groups (from the unique-gene functional breakdown)
groups <- list(
 "Chromatin / epigenetic" = c("DNMT1","SUZ12","AEBP2","RBBP4","KAT2B","KDM1B","KMT2E","TRIM24","NIPBL","SF3B1","RBM17","NUCKS1"),
 "DNA repair / DDR"       = c("PRKDC","XPA","FANCL","SHLD2","RAD54L2","PRIMPOL","NASP","CENPP","NEK9"),
 "Autophagy / lysosome"   = c("ATG12","WIPI1","NPC1","GGA2","ATP6V1H","PIP4P2","SPPL3"),
 "Ubiquitin / proteostasis" = c("UBE2N","UBE3C","UBQLN1","UBR3","TRIP12","USP37","SMURF1","RNF145")
)
gene_df <- do.call(rbind, lapply(names(groups), function(g)
    data.frame(gene=groups[[g]], grp=g, stringsAsFactors=FALSE)))

# pull logFC + FDR from each contrast
get_stats <- function(df, genes) df[match(genes, df$gene), c("logFC","adj.P.Val")]
sa <- get_stats(A, gene_df$gene); da <- get_stats(B, gene_df$gene)
hm <- rbind(
    data.frame(gene=gene_df$gene, grp=gene_df$grp, contrast="Senescence axis",
               logFC=sa$logFC, fdr=sa$adj.P.Val),
    data.frame(gene=gene_df$gene, grp=gene_df$grp, contrast="DAM axis",
               logFC=da$logFC, fdr=da$adj.P.Val)
)
hm <- hm[!is.na(hm$logFC), ]   # drop genes not tested in a contrast
hm$contrast <- factor(hm$contrast, levels=c("Senescence axis","DAM axis"))
hm$grp <- factor(hm$grp, levels=names(groups))
# order genes within group by senescence-axis logFC
ord <- A[match(gene_df$gene, A$gene),]; gene_df$saLFC <- ord$logFC
gene_order <- gene_df %>% arrange(grp, desc(saLFC)) %>% pull(gene)
hm$gene <- factor(hm$gene, levels=rev(gene_order))
hm$star <- ifelse(!is.na(hm$fdr) & hm$fdr<0.05, "*", "")

p <- ggplot(hm, aes(contrast, gene, fill=logFC)) +
    geom_tile(color="white", linewidth=0.5) +
    geom_text(aes(label=star), size=4, vjust=0.78, color="black") +
    scale_fill_gradient2(low="#377EB8", mid="white", high="#E41A1C", midpoint=0,
        name=expression(log[2]~"FC"), limits=max(abs(hm$logFC),na.rm=TRUE)*c(-1,1)) +
    facet_grid(grp~., scales="free_y", space="free_y", switch="y") +
    labs(title="Genome-maintenance genes: senescence axis vs DAM axis",
         subtitle="logFC per contrast | * = FDR<0.05 | senescence-axis-unique program",
         x=NULL, y=NULL) +
    theme_minimal(base_size=9) +
    theme(plot.title=element_text(size=11, face="bold"),
          plot.subtitle=element_text(size=7, color="grey45"),
          panel.grid=element_blank(),
          axis.text.x=element_text(size=9, face="bold"),
          axis.text.y=element_text(size=7),
          strip.text.y.left=element_text(size=7.5, face="bold", angle=0),
          strip.placement="outside",
          legend.key.width=unit(0.3,"cm"))

options(repr.plot.width=6, repr.plot.height=9); print(p)
save_figure(p, "heatmap_genome_maintenance_genes_Sen_vs_DAM_axis_microglia", width=6, height=9)
cat("\n✓ genome-maintenance heatmap done.\n")

---
## 08 · Export rankings for GSEA

**Why.** Writes the `.rnk` files and `gsea_config.json` that module 12 reads.
Handing off through files rather than a shared kernel keeps the R and Python
halves independent — module 12 can be re-run without refitting the models.

**Ranking metric.** Signed p-value, `-log10(p) * sign(logFC)`. It orders on
evidence rather than on effect size, which keeps low-expression genes with
large but noisy fold changes from dominating the head of the ranking.

In [ ]:
# writes: rankings + gsea_config.json  (see 05_gsea.ipynb)
GSEA_OUT <- file.path(PATHS$output_root, 'gsea_prerank')
dir.create(GSEA_OUT, recursive=TRUE, showWarnings=FALSE)
jsonlite::write_json(list(
  AXIS=AXIS, X_LAB=X_LAB, X_HEX=X_HEX, Y_TAG=Y_TAG, X_TAG=X_TAG,
  CONTRASTS=CONTRASTS, CONTRAST_HEADERS=CONTRAST_HEADERS,
  DBS=GSEA_DBS, FDR_SIG=GSEA_FDR_SIG,
  N_COMMON=GSEA_N_COMMON, N_UNIQUE=GSEA_N_UNIQUE,
  RIBO_STRIP=GSEA_RIBO_STRIP,
  OUT=GSEA_OUT, FIG=AXIS_FIG_DIR),
  file.path(GSEA_OUT, 'gsea_config.json'), auto_unbox=TRUE, pretty=TRUE)
cat('wrote gsea_config.json for axis', AXIS, '\n')